# NeuroGen Analysis

Visualize benchmark results and diagnostic metrics.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import glob

plt.style.use('seaborn-v0_8-whitegrid')
OUTPUTS = Path('outputs')

## Load Latest Benchmark

In [ ]:
csvs = sorted(OUTPUTS.glob('benchmark_*.csv'))
if not csvs:
    raise FileNotFoundError('No benchmark CSVs found. Run: uv run benchmark.py --compare "default,xavier" --seeds 5')
df = pd.read_csv(csvs[-1])
print(f'Loaded {csvs[-1].name}: {len(df)} runs, {df["method"].nunique()} methods')
df.head()

## val_bpb by Init Method

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
grouped = df.groupby('method')['val_bpb']
means = grouped.mean().sort_values()
stds = grouped.std().reindex(means.index)
means.plot(kind='bar', yerr=stds, ax=ax, capsize=4, color='steelblue', edgecolor='black')
ax.set_ylabel('val_bpb (lower is better)')
ax.set_title('Validation BPB by Initialization Method')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.savefig(OUTPUTS / 'val_bpb_comparison.png', dpi=150)
plt.show()

## Init Diagnostic Metrics

In [ ]:
diag_cols = ['init_head_diversity', 'init_block_diag_ratio', 'init_layer_similarity', 'init_weight_std']
available = [c for c in diag_cols if c in df.columns]

if available:
    fig, axes = plt.subplots(1, len(available), figsize=(4 * len(available), 5))
    if len(available) == 1:
        axes = [axes]
    for ax, col in zip(axes, available):
        grouped = df.groupby('method')[col]
        means = grouped.mean().sort_values()
        stds = grouped.std().reindex(means.index)
        means.plot(kind='bar', yerr=stds, ax=ax, capsize=3, color='coral', edgecolor='black')
        ax.set_title(col.replace('init_', ''))
        ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig(OUTPUTS / 'init_diagnostics.png', dpi=150)
    plt.show()
else:
    print('No init diagnostic columns found in data')

## Init Loss vs Final val_bpb

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for method, group in df.groupby('method'):
    ax.scatter(group['init_loss'], group['val_bpb'], label=method, s=60, alpha=0.8)
ax.set_xlabel('init_loss')
ax.set_ylabel('val_bpb')
ax.set_title('Does Better Init Translate to Better Final Model?')
ax.legend()
plt.tight_layout()
plt.savefig(OUTPUTS / 'init_vs_final.png', dpi=150)
plt.show()

## Live CA: Gradient Alignment Over Training

Parse `ca_grad_alignment` from run logs. Place log files in `outputs/` to analyze.

In [ ]:
import re

def parse_ca_alignment(log_path):
    """Extract ca_grad_alignment values from a training log."""
    steps, alignments = [], []
    step_pat = re.compile(r'step\s+(\d+)')
    align_pat = re.compile(r'ca_grad_alignment:\s+([\-\d.]+)')
    current_step = 0
    with open(log_path) as f:
        for line in f:
            m = step_pat.search(line)
            if m:
                current_step = int(m.group(1))
            m = align_pat.search(line)
            if m:
                steps.append(current_step)
                alignments.append(float(m.group(1)))
    return steps, alignments

log_files = sorted(OUTPUTS.glob('*.log')) + sorted(Path('.').glob('run.log'))
if log_files:
    fig, ax = plt.subplots(figsize=(8, 4))
    for lf in log_files:
        steps, aligns = parse_ca_alignment(lf)
        if aligns:
            ax.plot(steps, aligns, label=lf.stem, alpha=0.8)
    if ax.get_lines():
        ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
        ax.set_xlabel('Training Step')
        ax.set_ylabel('CA-Gradient Alignment')
        ax.set_title('CA Cooperation (+1) vs Competition (-1) with Gradient')
        ax.legend()
        plt.tight_layout()
        plt.savefig(OUTPUTS / 'ca_grad_alignment.png', dpi=150)
        plt.show()
    else:
        print('No ca_grad_alignment data found in logs')
else:
    print('No log files found. Run live CA experiments first.')

## Load All Benchmarks (Historical)

In [ ]:
all_csvs = sorted(OUTPUTS.glob('benchmark_*.csv'))
if len(all_csvs) > 1:
    all_df = pd.concat([pd.read_csv(c).assign(benchmark=c.stem) for c in all_csvs])
    print(f'Loaded {len(all_csvs)} benchmarks, {len(all_df)} total runs')
    display(all_df.groupby(['benchmark', 'method'])['val_bpb'].agg(['mean', 'std']).round(4))
else:
    print('Only one benchmark found. Run more to see historical trends.')